In [0]:
from pyspark.sql import functions as F

In [0]:
silver_table_name = (
    "global_economic_analytics.silver.macro_observation"
)

gold_table_name = (
    "global_economic_analytics.gold.country_year_macro"
)

print(silver_table_name)
print(gold_table_name)

In [0]:
df_silver = spark.table(silver_table_name)

print(df_silver.count())

In [0]:
display(
    df_silver
    .select(
        "indicator_code",
        "indicator_name"
    )
    .distinct()
    .orderBy("indicator_code")
)

In [0]:
df_gold_base = (
    df_silver
    .select(
        "country_code",
        "country_name",
        "year",
        "indicator_code",
        "indicator_value",
        "ingestion_timestamp"
    )
)

print(df_gold_base.count())

In [0]:
df_gold_pivot = (
    df_gold_base
    .groupBy(
        "country_code",
        "country_name",
        "year"
    )
    .pivot(
        "indicator_code",
        ["NY.GDP.MKTP.CD"]
    )
    .agg(
        F.first("indicator_value")
    )
    .withColumnRenamed(
        "NY.GDP.MKTP.CD",
        "gdp_current_usd"
    )
)

print(df_gold_pivot.count())

In [0]:
df_gold_pivot.printSchema()

In [0]:
display(
    df_gold_pivot
    .orderBy(F.col("year").desc())
    .limit(5)
)

In [0]:
df_gold_enriched = (
    df_gold_pivot
    .withColumn(
        "gdp_current_billion_usd",
        F.round(
            F.col("gdp_current_usd") / F.lit(1_000_000_000),
            2
        )
    )
)

display(
    df_gold_enriched
    .select(
        "country_code",
        "year",
        "gdp_current_usd",
        "gdp_current_billion_usd"
    )
    .orderBy(F.col("year").desc())
    .limit(5)
)

In [0]:
df_gold = (
    df_gold_enriched
    .withColumn(
        "country_year_key",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("country_code"),
                F.col("year").cast("string")
            ),
            256
        )
    )
)

print(df_gold.columns)

In [0]:
country_year_key_summary = (
    df_gold
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("country_year_key").alias("distinct_keys")
    )
    .first()
)

print(country_year_key_summary)

In [0]:
df_gold_organizado = df_gold.select(
    "country_year_key",
    "country_code",
    "country_name",
    "year",
    "gdp_current_usd",
    "gdp_current_billion_usd"
)

print(df_gold_organizado.columns)

In [0]:
gold_table_exists = spark.catalog.tableExists(
    gold_table_name
)

print(gold_table_exists)

In [0]:
(
    df_gold_organizado.write
    .format("delta")
    .saveAsTable(gold_table_name)
)

print("Tabela Gold criada com sucesso!")

In [0]:
gold_table_exists_after_write = spark.catalog.tableExists(
    gold_table_name
)

print(gold_table_exists_after_write)

In [0]:
gold_table_count = spark.table(
    gold_table_name
).count()

print(gold_table_count)

In [0]:
display(
    spark.table(gold_table_name)
    .orderBy(F.col("year").desc())
    .limit(5)
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE DETAIL {gold_table_name}
    """).select(
        "format",
        "name",
        "numFiles",
        "sizeInBytes"
    )
)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {gold_table_name}
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters"
    )
)

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {gold_table_name}
    IS 'Indicadores macroeconômicos organizados por país e ano para consumo analítico.'
""")

print("Comentário da tabela Gold adicionado com sucesso!")

In [0]:
gold_table_comment = spark.sql("""
    SELECT comment
    FROM global_economic_analytics.information_schema.tables
    WHERE table_schema = 'gold'
      AND table_name = 'country_year_macro'
""").first()["comment"]

print(gold_table_comment)

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    ALTER COLUMN country_year_key
    COMMENT 'Chave SHA-256 da combinação entre país e ano.'
""")

print("Coluna country_year_key documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    ALTER COLUMN country_code
    COMMENT 'Código ISO 3 do país.'
""")

print("Coluna country_code documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    ALTER COLUMN country_name
    COMMENT 'Nome do país conforme retornado pelo Banco Mundial.'
""")

print("Coluna country_name documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    ALTER COLUMN year
    COMMENT 'Ano de referência dos indicadores macroeconômicos.'
""")

print("Coluna year documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    ALTER COLUMN gdp_current_usd
    COMMENT 'Produto Interno Bruto em dólares correntes.'
""")

print("Coluna gdp_current_usd documentada com sucesso!")

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    ALTER COLUMN gdp_current_billion_usd
    COMMENT 'Produto Interno Bruto em bilhões de dólares correntes, arredondado para duas casas decimais.'
""")

print("Coluna gdp_current_billion_usd documentada com sucesso!")

In [0]:
display(
    spark.sql(f"""
        DESCRIBE TABLE {gold_table_name}
    """)
)

In [0]:
spark.sql(f"""
    ALTER TABLE {gold_table_name}
    SET TBLPROPERTIES (
        'project' = 'global_economic_analytics',
        'data_source' = 'World Bank API',
        'data_layer' = 'gold',
        'data_granularity' = 'one row per country and year'
    )
""")

print("Propriedades da tabela Gold adicionadas com sucesso!")

In [0]:
gold_table_properties = spark.sql(f"""
    SHOW TBLPROPERTIES {gold_table_name}
""")

display(
    gold_table_properties.filter(
        """
        key IN (
            'project',
            'data_source',
            'data_layer',
            'data_granularity'
        )
        """
    )
)

In [0]:
gold_source_view = "vw_country_year_macro_source"

df_gold_organizado.createOrReplaceTempView(
    gold_source_view
)

print(gold_source_view)

In [0]:
gold_source_count = spark.table(
    gold_source_view
).count()

print(gold_source_count)

In [0]:
spark.sql(f"""
    MERGE INTO {gold_table_name} AS target
    USING {gold_source_view} AS source
        ON target.country_year_key = source.country_year_key

    WHEN MATCHED THEN UPDATE SET
        target.country_code = source.country_code,
        target.country_name = source.country_name,
        target.year = source.year,
        target.gdp_current_usd = source.gdp_current_usd,
        target.gdp_current_billion_usd = source.gdp_current_billion_usd

    WHEN NOT MATCHED THEN INSERT (
        country_year_key,
        country_code,
        country_name,
        year,
        gdp_current_usd,
        gdp_current_billion_usd
    )
    VALUES (
        source.country_year_key,
        source.country_code,
        source.country_name,
        source.year,
        source.gdp_current_usd,
        source.gdp_current_billion_usd
    )
""")

print("MERGE Gold executado com sucesso!")

In [0]:
gold_count_after_merge = spark.table(
    gold_table_name
).count()

print(gold_count_after_merge)

In [0]:
display(
    spark.sql(f"""
        DESCRIBE HISTORY {gold_table_name}
    """)
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
    .orderBy(F.col("version").desc())
    .limit(1)
)

In [0]:
gold_view_removed = spark.catalog.dropTempView(
    gold_source_view
)

print(gold_view_removed)

In [0]:
gold_source_view_exists = spark.catalog.tableExists(
    gold_source_view
)

print(gold_source_view_exists)

In [0]:
gold_final_validation = (
    spark.table(gold_table_name)
    .agg(
        F.count("*").alias("total_records"),
        F.countDistinct("country_year_key").alias("distinct_keys"),
        F.countDistinct("country_code").alias("country_count"),
        F.min("year").alias("minimum_year"),
        F.max("year").alias("maximum_year"),
        F.sum(
            F.col("gdp_current_usd").isNull().cast("int")
        ).alias("null_gdp_count")
    )
    .first()
)

print(gold_final_validation)

In [0]:
assert gold_final_validation["total_records"] == 25
assert gold_final_validation["distinct_keys"] == 25
assert gold_final_validation["country_count"] == 1
assert gold_final_validation["minimum_year"] == 2000
assert gold_final_validation["maximum_year"] == 2024
assert gold_final_validation["null_gdp_count"] == 0

print("Validação final da tabela Gold concluída com sucesso!")